## 0. Path bootstrap

In [1]:
import sys
from pathlib import Path

# Ensure this directory is on the path so all local modules are importable.
NOTEBOOK_DIR = Path(r"C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/MCC_contrained_decoding")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Working dir : {NOTEBOOK_DIR}")

Working dir : C:\Users\vimal\OneDrive\Documents\Uni\BTP\User-Adaptive-XAI\MCC_contrained_decoding


## 1. Imports

In [2]:
import warnings

from IPython.display import display
import numpy as np
import pandas as pd

from config import (
    # Experiment knobs
    ABLATION_MODE,
    EXPERIMENT_RESULTS_PATH,
    EXPERIMENT_TAG,
    INPUT_TEXT,
    USER_CATEGORY,
    # XAI method  ← change XAI_METHOD in config.py to swap
    XAI_METHOD,
    XAI_NUM_FEATURES,
    XAI_NUM_SAMPLES,
    CLASS_NAMES,
    # LLM / decoding
    LAMBDA_MAP,
    NUM_BEAMS,
    USE_CONSTRAINED_DECODING,
    # Ontology
    TOP_LIME_FEATURES,
)
from constrained_decoding import ReadabilityBeamGenerator
from model_loaders import load_classifier, load_llm, load_ontology_model
from pipeline_helpers import (
    SYSTEM_PROMPT,
    build_prompt,
    enrich_with_ontology,
    generate_explanation,
    lime_coverage,
    ontology_hit_rate,
    predict_class,
    readability_metrics,
    run_xai,
)

warnings.filterwarnings("ignore")
print("✅ Imports complete.")

✅ Imports complete.


## 2. Experiment configuration (read-only — edit config.py)

In [3]:
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  EXPERIMENT CONFIG")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Tag                  : {EXPERIMENT_TAG}")
print(f"  XAI method           : {XAI_METHOD}")
print(f"  User category        : {USER_CATEGORY}")
print(f"  Ablation mode        : {ABLATION_MODE}")
print(f"  Constrained decoding : {USE_CONSTRAINED_DECODING}")
print(f"  Lambda value         : {LAMBDA_MAP.get(USER_CATEGORY, 'N/A')}")
print(f"  Num beams            : {NUM_BEAMS}")
print(f"  Output path          : {EXPERIMENT_RESULTS_PATH}")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EXPERIMENT CONFIG
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tag                  : expert_normal_ig
  XAI method           : IG
  User category        : EXPERT
  Ablation mode        : normal
  Constrained decoding : False
  Lambda value         : 0.05
  Num beams            : 4
  Output path          : Ablation_Study_WO_CD\expert_normal_ig.csv
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 3. Input text

In [4]:
if INPUT_TEXT is not None:
    input_text = INPUT_TEXT.strip()
    print("[Input] Using text from config.py INPUT_TEXT.")
else:
    data_path = NOTEBOOK_DIR / "test_data.txt"
    with open(data_path, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    input_text = lines[0]
    print("[Input] INPUT_TEXT is None — using first line of test_data.txt.")

print(f"\nText ({len(input_text)} chars):")
print(input_text[:400] + ("…" if len(input_text) > 400 else ""))

[Input] Using text from config.py INPUT_TEXT.

Text (462 chars):
Endometriosis associated with massive ascites and absence of pelvic peritoneum. Although massive ascites associated with endometriosis has been reported in rare cases, this patient was also noted to have massive destruction of the pelvic peritoneum. Failure of medical suppression necessitated total abdominal hysterectomy and bilateral salpingo-oophorectomy. Several months after surgery ascites res…


## 4. Load models (classifier, ontology, LLM)

This is the slow step. All models are loaded once here and reused across stages.
NER is not loaded — LIME runs directly on raw text.

In [5]:
# ── Classifier ────────────────────────────────────────────────────────────────
clf_model, clf_pipeline = load_classifier()

# ── Ontology ──────────────────────────────────────────────────────────────────
ontology = load_ontology_model()

# ── LLM ───────────────────────────────────────────────────────────────────────
llm_tokenizer, llm_model = load_llm()

# ── Constrained-decoding generator ────────────────────────────────────────────
generator = None
if USE_CONSTRAINED_DECODING:
    generator = ReadabilityBeamGenerator(
        model=llm_model,
        tokenizer=llm_tokenizer,
        num_beams=NUM_BEAMS,
    )
    print(f"[CD] Constrained decoding enabled — λ={LAMBDA_MAP.get(USER_CATEGORY, '?')}, beams={NUM_BEAMS}")
else:
    print("[CD] Constrained decoding disabled (greedy/sample mode).")

print("\n✅ All models loaded.")

[Loader] Loading classifier from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Models/my_medical_model' …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Loader] Classifier ready.

[Ontology] Loading from 'C:/Users/vimal/OneDrive/Documents/Uni/BTP/User-Adaptive-XAI/Ontology/doid.owl' …
[Ontology] Loaded successfully.
[Ontology] Stats → classes: 14493, est. max depth: 15
[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[Loader] LLM ready.

[CD] Constrained decoding disabled (greedy/sample mode).

✅ All models loaded.


## Stage 1 — LIME Feature Attribution

Raw text is fed directly to LIME (no NER merging). Features are single words.

In [6]:
print(f"[Stage 1] Running {XAI_METHOD} feature attribution on raw text …")
xai_features = run_xai(
    method=XAI_METHOD,
    text=input_text,
    clf_model=clf_model,
    clf_pipeline=clf_pipeline,
    class_names=CLASS_NAMES,
    num_features=XAI_NUM_FEATURES,
    num_samples=XAI_NUM_SAMPLES,
)

# Alias so that Stages 2–4 see the same variable name as before.
lime_features = xai_features

print(f"\n[Stage 1] Top {len(lime_features)} {XAI_METHOD} features:")
for word, score in lime_features:
    print(f"  {word:25s}  score={score:+.4f}")

print("\n✅ Stage 1 complete.")

[Stage 1] Running IG feature attribution on raw text …

[Stage 1] Top 6 IG features:
  ascites                    score=+0.2454
  endometriosis              score=+0.2021
  peritoneum                 score=+0.1096
  massive                    score=+0.0978
  oophorectomy               score=+0.0946
  pelvic                     score=+0.0921

✅ Stage 1 complete.


## Stage 2 — Ontology Enrichment

In [7]:
print("[Stage 2] Classifying text …")
predicted_class, confidence = predict_class(input_text, clf_pipeline)
print(f"  Predicted class : {predicted_class}")
print(f"  Confidence      : {confidence:.4f}")

print("\n[Stage 2] Mapping LIME features to ontology ancestors …")
feature_data = enrich_with_ontology(
    lime_features=lime_features,
    ontology=ontology,
    user_category=USER_CATEGORY,
    ablation_mode=ABLATION_MODE,
)

hit_words = [f["feature_word"] for f in feature_data]
print(f"  Ontology hits : {hit_words} ({len(hit_words)}/{len(lime_features[:TOP_LIME_FEATURES])} features)")
for f in feature_data:
    print(f"    {f['feature_word']:20s} → {f['ancestors']}")

print("\n✅ Stage 2 complete.")

[Stage 2] Classifying text …
  Predicted class : Digestive system diseases
  Confidence      : 0.6624

[Stage 2] Mapping LIME features to ontology ancestors …
  Ontology hits : ['ascites', 'endometriosis', 'peritoneum'] (3/6 features)
    ascites              → ['symptom', 'abdominal symptom', 'ascites']
    endometriosis        → ['disease of anatomical entity', 'reproductive system disease', 'female reproductive system disease', 'endometriosis']
    peritoneum           → ['mesoderm-derived structure', 'multi-tissue structure', 'serous membrane', 'peritoneum']

✅ Stage 2 complete.


## Stage 3 — LLM Explanation Generation

In [8]:
print("[Stage 3] Building prompt …")
prompt = build_prompt(
    text=input_text,
    predicted_class=predicted_class,
    feature_data=feature_data,
    user_category=USER_CATEGORY,
)

if USE_CONSTRAINED_DECODING:
    lam = LAMBDA_MAP.get(USER_CATEGORY, LAMBDA_MAP["EXPERT"])
    print(f"[Stage 3] Generating with constrained decoding (λ={lam}, beams={NUM_BEAMS}) …")
else:
    print("[Stage 3] Generating with standard sampling …")

explanation = generate_explanation(
    text=input_text,
    predicted_class=predicted_class,
    feature_data=feature_data,
    user_category=USER_CATEGORY,
    tokenizer=llm_tokenizer,
    model=llm_model,
    generator=generator,
)

print("\n── Generated explanation ─────────────────────────────────────")
print(explanation)
print("─────────────────────────────────────────────────────────────")
print("\n✅ Stage 3 complete.")

[Stage 3] Building prompt …
[Stage 3] Generating with standard sampling …

── Generated explanation ─────────────────────────────────────
The model's prediction that the abstract belongs to the digestive system diseases category is driven by several key words and phrases. Specifically, the presence of "massive ascites," "endometriosis," and "absence of pelvic peritoneum" are crucial factors. 

Firstly, "massive ascites" indicates an accumulation of fluid in the abdomen, which can lead to various complications such as pressure on internal organs and impaired blood flow. This condition often requires surgical intervention.

Secondly, "endometriosis" refers to tissue similar to the lining inside the uterus growing outside it, commonly found in women. It can cause pain, infertility, and other symptoms. In this case, the patient had both massive ascites and destruction of the pelvic peritoneum due to endometriosis, suggesting a complex interplay between these two conditions.

Lastly, "absen

## Stage 4 — Readability & Faithfulness Metrics

In [9]:
print("[Stage 4] Computing metrics …")

read_metrics = readability_metrics(explanation)
cov = lime_coverage(explanation, feature_data)
hit = ontology_hit_rate(feature_data)

result = {
    "experiment_tag":        EXPERIMENT_TAG,
    "xai_method":            XAI_METHOD,
    "user_category":         USER_CATEGORY,
    "ablation_mode":         ABLATION_MODE,
    "constrained_decoding":  USE_CONSTRAINED_DECODING,
    "lambda":                LAMBDA_MAP.get(USER_CATEGORY, None),
    "ner_merging":           False,
    "predicted_class":       predicted_class,
    "confidence":            confidence,
    "text_snippet":          input_text[:120] + "…",
    "explanation":           explanation,
    "lime_coverage":         cov,
    "ontology_hit_rate":     hit,
    **read_metrics,
}

df = pd.DataFrame([result])

print("\n── Metrics ───────────────────────────────────────────────────")
metric_cols = [
    "xai_method", "user_category", "ablation_mode", "constrained_decoding", "ner_merging",
    "predicted_class", "confidence",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate",
]
pd.set_option("display.max_colwidth", 40)
display(df[metric_cols])

print("\n✅ Stage 4 complete.")

[Stage 4] Computing metrics …

── Metrics ───────────────────────────────────────────────────


,xai_method,user_category,ablation_mode,constrained_decoding,ner_merging,predicted_class,confidence,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,IG,EXPERT,normal,False,False,Digestive system diseases,0.6624,30.847859,12.900463,13.624085,1.0,1.0



✅ Stage 4 complete.


## 5. Save final results

In [10]:
EXPERIMENT_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(EXPERIMENT_RESULTS_PATH, index=False)
print(f"✅ Results saved → '{EXPERIMENT_RESULTS_PATH}'")

print("\n── Full explanation ──────────────────────────────────────────")
print(f"  Class      : {predicted_class} (conf={confidence:.4f})")
print(f"  User       : {USER_CATEGORY}")
print(f"  Ablation   : {ABLATION_MODE}")
print(f"  Tag        : {EXPERIMENT_TAG}")
print("─────────────────────────────────────────────────────────────")
print(explanation)

✅ Results saved → 'Ablation_Study_WO_CD\expert_normal_ig.csv'

── Full explanation ──────────────────────────────────────────
  Class      : Digestive system diseases (conf=0.6624)
  User       : EXPERT
  Ablation   : normal
  Tag        : expert_normal_ig
─────────────────────────────────────────────────────────────
The model's prediction that the abstract belongs to the digestive system diseases category is driven by several key words and phrases. Specifically, the presence of "massive ascites," "endometriosis," and "absence of pelvic peritoneum" are crucial factors. 

Firstly, "massive ascites" indicates an accumulation of fluid in the abdomen, which can lead to various complications such as pressure on internal organs and impaired blood flow. This condition often requires surgical intervention.

Secondly, "endometriosis" refers to tissue similar to the lining inside the uterus growing outside it, commonly found in women. It can cause pain, infertility, and other symptoms. In this c